Import & Start Spark


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Ecommerce_BigData_Project") \
    .getOrCreate()

Load Dataset



In [ ]:
df = spark.read.csv("amazon_sales_dataset.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

Data Cleaning

In [ ]:
# Remove null values
df = df.dropna()

# Remove duplicates
df = df.dropDuplicates()

Feature Engineering

In [ ]:
from pyspark.sql.functions import col, month, year

# Create Revenue column
df = df.withColumn("Revenue", col("Price") * col("Quantity"))

# Extract Month & Year
df = df.withColumn("Month", month(col("Date")))
df = df.withColumn("Year", year(col("Date")))

Exploratory Data Analysis (EDA)

In [ ]:
#Top Selling products
df.groupBy("Product") \
  .sum("Quantity") \
  .orderBy("sum(Quantity)", ascending=False) \
  .show()

In [ ]:
#Revenue by Category
df.groupBy("Category") \
  .sum("Revenue") \
  .orderBy("sum(Revenue)", ascending=False) \
  .show()

In [ ]:
#City-wise Revenue
df.groupBy("City") \
  .sum("Revenue") \
  .orderBy("sum(Revenue)", ascending=False) \
  .show()

In [ ]:
#Monthly Sales Trend
df.groupBy("Month") \
  .sum("Revenue") \
  .orderBy("Month") \
  .show()

Machine Learning (PySpark ML)

In [ ]:
#Import ML Libraries
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
df_ml = df.select("Price", "Quantity", "Month", "Revenue")

In [ ]:
assembler = VectorAssembler(
    inputCols=["Price", "Quantity", "Month"],
    outputCol="features"
)

data = assembler.transform(df_ml)
data = data.select("features", "Revenue")

In [ ]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Revenue",
    numTrees=50
)

model = rf.fit(train_data)

In [ ]:
predictions = model.transform(test_data)

predictions.select("features", "Revenue", "prediction").show()

In [ ]:
#Evaluate model 
evaluator_r2 = RegressionEvaluator(
    labelCol="Revenue",
    predictionCol="prediction",
    metricName="r2"
)

evaluator_rmse = RegressionEvaluator(
    labelCol="Revenue",
    predictionCol="prediction",
    metricName="rmse"
)

r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)

print("R2 Score:", r2)
print("RMSE:", rmse)

Save Results (Optional but IMPRESSIVE)

In [ ]:
predictions.write.csv("output_predictions", header=True)